### setup

In [1]:
import os
import glob
import time
import fastf1
import logging
import warnings
import pandas as pd
from datetime import datetime

# enable cache
fastf1.Cache.enable_cache("cache")

### get schedule for all years avaliable 
* only need to run once

In [12]:
# # initialize list to hold data
# all_schedules = []

# # iteratively get one year at a time
# for year in range(2018, 2027): 
#     try:
#         # get schedule
#         schedule = fastf1.get_event_schedule(year)
#         # uncomment if you want to save schedule by year
#         # schedule.to_csv(f"data/event_schedule_{year}.csv", index=False)

#         # add column for year and add schedule to list of schedules
#         temp = schedule.copy()
#         temp["Season"] = year
#         all_schedules.append(temp)
        
#         print(f"saved {year}")

#     except Exception as e:
#         print(f"skipping {year}: {e}")

# # join dataframes for all years together and save to csv
# full_schedule = pd.concat(all_schedules, ignore_index=True)
# full_schedule.to_csv("data/event_schedule.csv", index=False)

### get data by year
* need to pull in chunks because the API has a limit of 500 calls per hour

In [25]:
# # read saved schedule data
# schedule = pd.read_csv("data/event_schedule.csv")

# # adjust year
# year = 2026

# # filter schedule by year
# schedule["EventDate"] = pd.to_datetime(schedule["EventDate"])
# schedule = schedule[schedule["EventDate"].dt.year == year]

# # iterate over scheduled events
# for _, row in schedule.iterrows():
#     # get event name
#     event_name = row["EventName"]
#     event_date = row["EventDate"]

#     try:
#         # get data by year and event
#         session = fastf1.get_session(year, event_name, "Q")
#         session.load()

#         # get tables
#         laps = session.laps
#         results = session.results
#         weather = session.weather_data
        
#         # add event name and date to all tables
#         laps['EventName'] = event_name
#         laps['EventDate'] = event_date
#         results['EventName'] = event_name
#         results['EventDate'] = event_date
#         weather['EventName'] = event_name
#         weather['EventDate'] = event_date
         
#         fname = event_name.replace(" ", "_")
#         # save tables to data folder (save in chunks in case API limit is hit)
#         laps.to_csv(f"data/{year}_{fname}_laps.csv", index=False)
#         results.to_csv(f"data/{year}_{fname}_results.csv", index=False)
#         weather.to_csv(f"data/{year}_{fname}_weather.csv", index=False)

#         print(f"saved {event_name}")

#     except Exception as e:
#         print(f"skipped {event_name}: {e}")

### consolidate files into single file
* if run multiple times might create duplicates if a year had to be run more than once above

In [27]:
# file_groups = {"laps": glob.glob(os.path.join("data", "*_laps.csv")),
#                "results": glob.glob(os.path.join("data", "*_results.csv")),
#                "weather": glob.glob(os.path.join("data", "*_weather.csv"))}

# for kind, files in file_groups.items():
#     master_path = os.path.join(data_dir, f"all_{kind}.csv")
#     dfs = []
    
#     if os.path.exists(master_path):
#         dfs.append(pd.read_csv(master_path))

#     for f in files:
#         if os.path.abspath(f) == os.path.abspath(master_path):
#             continue
#         dfs.append(pd.read_csv(f))

#     combined = pd.concat(dfs, ignore_index=True)
#     combined.to_csv(master_path, index=False)

#     for f in files:
#         if os.path.abspath(f) != os.path.abspath(master_path):
#             os.remove(f)

#     print(f"consolidated {kind} into {master_path}")

### get throttle percentage data

In [2]:
schedule = pd.read_csv("data/event_schedule.csv")
schedule["EventDate"] = pd.to_datetime(schedule["EventDate"])

In [ ]:
rows = []

for _, row in schedule.iterrows():
    year = row["EventDate"].year
    event_name = row["EventName"]
    event_date = row["EventDate"]

    session = fastf1.get_session(year, event_name, "Q")
    session.load()
    fastest_lap = session.laps.pick_fastest()
    telemetry = fastest_lap.get_telemetry()
    full_throttle_pct = (telemetry["Throttle"] >= 98).mean() * 100

    rows.append({"Year": year, "EventName": event_name, "EventDate": event_date,
                 "Driver": fastest_lap["Driver"], "Team": fastest_lap["Team"],
                 "LapTime": fastest_lap["LapTime"], "FullThrottlePct": full_throttle_pct})
    
throttle_df = pd.DataFrame(rows)
throttle_df.to_csv("data/throttle_pct.csv", index=False)